In [1]:
import pandas as pd

df = pd.read_parquet("../data/raw/issues_sample.parquet")

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

Rows: 5000
Columns: ['issue_id', 'number', 'title', 'body', 'state', 'created_at', 'comments_count', 'source']


,issue_id,number,title,body,state,created_at,comments_count,source
0,26,26,Unique index behavior inconsistent,I think I found a bug in the unique index thin...,closed,2009-04-21T06:37:22-04:00,7,jira
1,52,52,dbs count == 0 fails in pair3 test,count for port: 44465 is not 0 is: 1\nSat May ...,closed,2009-05-18T06:26:34-04:00,0,jira
2,110,110,range query breaks on nested documents after c...,reported by raphael marvie on the list:\n\n>>>...,closed,2009-06-17T08:00:44-04:00,1,jira
3,117,117,limit() is returning less than it should,Ian tried to do \n\ndb.blog.posts.find({relate...,closed,2009-06-25T09:20:28-04:00,1,jira
4,133,133,Feature request: Improved error messaging,Instead of:\n\nArray\n(\n [err] => E11000 d...,closed,2009-07-07T18:16:37-04:00,2,jira


In [2]:
# How many from each source?
print("Source counts:")
print(df["source"].value_counts())

# How long are the ticket bodies? (in characters)
# We combine title + body into the text we'll eventually classify.
df["text"] = df["title"].fillna("") + "\n\n" + df["body"].fillna("")

print("\nText length (characters):")
print(df["text"].str.len().describe())

Source counts:
source
jira      2500
github    2500
Name: count, dtype: int64

Text length (characters):
count      5000.000000
mean       1242.316000
std        6945.617277
min           4.000000
25%         138.000000
50%         337.000000
75%         851.000000
max      277824.000000
Name: text, dtype: float64


In [3]:
# Show a few full tickets so we can read them properly.
# pandas truncates by default; this widens the display temporarily.
pd.set_option("display.max_colwidth", 1000)

# Pick 5 random tickets (random_state makes it reproducible, so same 5 each run)
sample = df.sample(5, random_state=42)

for _, row in sample.iterrows():
    print("=" * 70)
    print(f"SOURCE: {row['source']}  |  STATE: {row['state']}")
    print(f"TITLE: {row['title']}")
    print(f"BODY:\n{row['body'][:800]}")   # first 800 chars of body
    print()

SOURCE: jira  |  STATE: closed
TITLE: server logfile lines should include an indication of which level of verbosity caused them to be logged
BODY:
We want to be able to grep for important log events, or to exclude unimportant log events when using the -vvvv type logging verbosity settings.  Therefore, please include some kind of indicator at the head of log lines that indicates the level of verbosity that caused that log line to be printed.


SOURCE: github  |  STATE: closed
TITLE: kv rest api: add remaining range tests
BODY:


SOURCE: github  |  STATE: closed
TITLE: Add benchmark tests for MVCC merge operator.
BODY:


SOURCE: jira  |  STATE: closed
TITLE: Assertion Error:  bsonobjbuilder.h:127
BODY:
During a test, was bombarding the mongos with lots of updates (15K / second).
MongoS running on my desktop, sharded replicated server running at datacenter.
After running for a minute or two, saw one of the shards die (both primary and replica mongod's).
launched mongo, did db.metricValue.

In [4]:
# Load the candidates the pre-filter produced
cand = pd.read_parquet("../data/candidates/candidates.parquet")
print(f"Candidates: {len(cand)}")

# Which keywords are firing most? (this reveals noisy patterns)
from collections import Counter
kw_counter = Counter()
for hits in cand["keyword_hits"]:
    if hits:
        for kw in hits.split(", "):
            kw_counter[kw] += 1

print("\nTop keywords firing:")
for kw, n in kw_counter.most_common(20):
    print(f"  {n:4d}  {kw}")

Candidates: 222

Top keywords firing:
    47  authentication
    31  password
    18  tracking
    16  profiling
    11  credentials
     5  analytics
     4  encryption
     3  processor
     2  privacy
     2  purge
     2  data transfer
     2  user data
     1  scoring
     1  audit log
     1  access control
     1  third party
     1  user information


In [5]:
# Are GitHub tickets emptier than Jira ones? Check body lengths by source.
df_raw = pd.read_parquet("../data/raw/issues_sample.parquet")
df_raw["body_len"] = df_raw["body"].fillna("").str.len()

print("Median body length by source:")
print(df_raw.groupby("source")["body_len"].median())

print("\nShare of EMPTY bodies by source:")
print(df_raw.groupby("source")["body_len"].apply(lambda s: (s == 0).mean()))

Median body length by source:
source
github    145.0
jira      523.0
Name: body_len, dtype: float64

Share of EMPTY bodies by source:
source
github    0.1868
jira      0.0448
Name: body_len, dtype: float64


In [6]:
cand = pd.read_parquet("../data/candidates/candidates.parquet")

# How many candidates tripped ONLY exposure (no keyword, no fuzzy)?
only_exposure = cand[(cand["keyword_hits"] == "") & (cand["fuzzy_hits"] == "") & (cand["exposure_hits"] != "")]
print(f"Exposure-only candidates: {len(only_exposure)} / {len(cand)}")

# What kinds of exposure are firing?
from collections import Counter
exp_counter = Counter()
for hits in cand["exposure_hits"]:
    if hits:
        for h in hits.split(", "):
            # group ip:1.2.3.4 as just "ip"
            key = "ip" if h.startswith("ip:") else h
            exp_counter[key] += 1
print("\nExposure types firing:")
for k, n in exp_counter.most_common():
    print(f"  {n:4d}  {k}")

# Peek at a few exposure-only candidates to judge if they're real
pd.set_option("display.max_colwidth", 300)
print("\n--- Sample exposure-only candidates ---")
for _, row in only_exposure.sample(min(5, len(only_exposure)), random_state=1).iterrows():
    print(f"\n[{row['repo']}] {row['title']}")
    print(f"   exposure: {row['exposure_hits']}")
    print(f"   body: {row['body'][:200]}")

Exposure-only candidates: 309 / 377

Exposure types firing:
   186  ip
   119  email
    13  hostname

--- Sample exposure-only candidates ---

[github__microsoft__WSL] Bugcheck on VfsLookupDirectoryEntryInode TRK:0441000864
   exposure: email
   body: On March 21, 2017 I sent a report about a bugcheck to secure@microsoft.com. It had complete instructions on how to reproduce and the WinDbg `!analyze -v` output.

The report was assigned the number 

[github__microsoft__WSL] SystemCTL doesnt work
   exposure: email
   body: This bug-tracker is monitored by developers and other technical types.  We like detail!  So please use this form and tell us, concisely but precisely, what's up.  Please fill out ALL THE FIELDS!

If

[github__microsoft__WSL] can't traceroute with bash on windows / sockets broken
   exposure: ip:74.125.21.147
   body: probably a duplicate issue but....

trying to run traceroute after ifconfig failed, 

traceroute www.google.com
traceroute to www.google.com (74.125.21.1

In [2]:
import pandas as pd

synth = pd.read_parquet("../data/labeled/synthetic.parquet")
pd.set_option("display.max_colwidth", 600)

for kind in ["trigger", "exposure", "hard_neg"]:
    print("=" * 75)
    print(f"KIND: {kind}")
    print("=" * 75)
    sample = synth[synth["synth_kind"] == kind].sample(2, random_state=7)
    for _, row in sample.iterrows():
        print(f"\nLABEL: {row['label']}")
        print(row["text"][:500])
        print("-" * 40)

KIND: trigger

LABEL: relevant
support tool: bulk lookup by email returns full account objects - need to scope down

the internal support tool (`tools.internal/support/lookup`) currently lets anyone with a support role do a bulk CSV upload of email addresses and get back full account JSON for each match.

what 'full account JSON' includes right now:
- all user table fields incl. `date_of_birth`, `phone_number`, `hashed_password` (bcrypt, but still), `failed_login_count`, `last_login_ip`
- all linked addresses (shipping + bill
----------------------------------------

LABEL: relevant
Snowflake — why are raw webhook payloads landing in DW?

Doing some cleanup in `RAW_INGEST` schema and found a table `stripe_webhook_events` with ~2 years of unprocessed Stripe payloads. These include the full JSON body from Stripe which contains:
- `billing_details.name`
- `billing_details.email`
- `billing_details.address` (full)
- `card.last4`, `card.exp_month`, `card.exp_year`

Who set this up? There's 